# Transfit Tutorial

This notebook is an end-to-end guide for forward modeling, fitting, result inspection, and plotting.
All public time inputs in TransFit are interpreted as observer-frame days relative to a reference epoch, not raw JD/MJD values.
For multi-band calculations, TransFit always computes model `f_nu` first, then applies extinction on the model side, and only then converts to the requested observation space (`flux`, `AB mag`, or `Vega mag`).

Contents:
1. Model list and parameter meanings
2. Forward-model inputs (`z`, explicit distance, `filters`, and `extinction`)
3. Forward-model quick checks (bolometric + multi-band)
4. Bolometric fitting with `tf.fit_bol(...)`
5. Understanding `res` (`FitResult`)
6. Plot bolometric fit and posterior corner
7. Multi-band fitting with `tf.fit_multiband(...)`
8. Plot multi-band fit and posterior corner
9. Custom plotting with direct prediction helpers
10. Save/load fit results


In [ ]:
from pathlib import Path
import warnings
import sys

# Ensure the local TransFit repo is imported, even if the notebook is run from examples/
cwd = Path.cwd().resolve()
repo_root = None
for cand in (cwd, cwd.parent):
    if (cand / "transfit").is_dir():
        repo_root = cand
        break
if repo_root is None:
    raise RuntimeError("Could not locate local transfit package root.")

sys.path.insert(0, str(repo_root))

# Force reload from the local repo to avoid stale modules in a long-lived notebook kernel
for k in list(sys.modules.keys()):
    if k == "transfit" or k.startswith("transfit."):
        del sys.modules[k]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import transfit as tf

assert str(Path(tf.__file__).resolve()).startswith(str(repo_root)), tf.__file__
print("Using transfit from:", Path(tf.__file__).resolve())

warnings.filterwarnings(
    'ignore',
    message='Using a user-supplied explicit distance that differs from the Planck15 luminosity distance implied by z by more than 5%.*',
    category=UserWarning,
)

np.random.seed(123)

# Force plain matplotlib defaults (no background style / no grid)
plt.style.use("default")
plt.rcParams["axes.grid"] = False
plt.rcParams["axes.facecolor"] = "white"


def find_data_dir() -> Path:
    candidates = [
        Path.cwd() / "examples" / "data",
        Path.cwd() / "data",
        Path.cwd().parent / "examples" / "data",
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("Cannot find examples/data directory from current working directory.")


DATA_DIR = find_data_dir()
print("Data directory:", DATA_DIR)



## 1. Models and Parameter Order

Recommended public model keys:
- `nickel`
- `magnetar`
- `magnetar_ni`

Current interface note:
- the public model keys above are the supported model names.
- this tutorial uses the current public model names only

Forward-model note:
- `nickel` and `magnetar` can still apply internal legacy defaults in some forward helpers
- in this notebook we pass the full parameter dictionaries explicitly so the API usage is unambiguous

Fit note:
- `t_shift` is an optional fitted parameter
- `fit_bol(...)` keeps `T_floor` internal and does not fit it directly

Likelihood convention:
`model(t_obs + t_shift)`
which aligns model-time with observed-time.

Positive `t_shift` shifts the model curve to earlier observed times.



In [ ]:
MODEL_KEYS = ["nickel", "magnetar", "magnetar_ni"]

for m in MODEL_KEYS:
    print(f"{m:12s}: {tf.model_param_names(m)}")



### Parameter Meaning and Unit

| Parameter | Meaning | Unit |
|---|---|---|
| `M_ej` | Ejecta mass | `M_sun` |
| `v_ej` | Ejecta velocity scale | `1e9 cm s^-1` |
| `M_Ni` | Nickel mass | `M_sun` |
| `E_Th_in` | Initial thermal energy scale | `1e49 erg` |
| `R_0` | Initial radius scale | `R_sun` |
| `x_Ni` | Heating radius fraction | dimensionless |
| `kappa` | Optical opacity | `cm^2 g^-1` |
| `kappa_gamma` | Gamma opacity | `cm^2 g^-1` |
| `P_ms` | Magnetar initial spin period | `ms` |
| `B14` | Magnetar magnetic field scale | `1e14 G` |
| `T_floor` | Temperature floor | `K` |
| `t_shift` | Model-observation time offset | `day` |

For bolometric fitting, `T_floor` is kept internal by `tf.fit_bol(...)` and is not sampled directly.



## 2. Set Forward-Model Inputs

- Use `z` for observer-frame redshift handling.
- Use `distance_modulus` when you want to override the cosmology-derived luminosity distance.
- Use `filters` directly for multi-band calculations; it can be a legacy `{band: nu_eff_hz}` map or a built-in preset map such as `{"B": "johnson_cousins.B"}`.
- Set `mag_system="ab"` or `mag_system="vega"` when `y_kind="mag"`.
- Pass direct `A_band` values such as `extinction={"B": 0.12, "V": 0.09}` or structured dust components such as `extinction={"mw": {"ebv": 0.04, "rv": 3.1, "law": "odonnell94"}}`.
- Internally, the multi-band pipeline is always `model state -> f_nu -> extinction on the model -> output space`.
- Even in forward-model helpers, time inputs are interpreted as observer-frame days relative to a reference epoch.


In [ ]:
# Example redshift, explicit distance, filters, and extinction inputs
z = 0.001728
distance_modulus = 29.84
filters = {
    "B": "johnson_cousins.B",
    "V": "johnson_cousins.V",
    "R": "johnson_cousins.R",
    "I": "johnson_cousins.I",
}
extinction = {
    "mw": {"ebv": 0.04, "rv": 3.1, "law": "odonnell94"},
}

z, distance_modulus, filters, extinction


## 3. Forward-model Quick Checks

These checks are useful before fitting: verify model shape and approximate scale.


In [ ]:
# Bolometric forward example (nickel)
params_nickel_bol = {
    "M_ej": 3.0,
    "v_ej": 1.0,
    "E_Th_in": 1.5,
    "M_Ni": 0.08,
    "R_0": 120.0,
    "x_Ni": 0.2,
    "kappa": 0.12,
    "kappa_gamma": 0.03,
}
bol = tf.lightcurve_bol(
    model="nickel",
    params=params_nickel_bol,
    z=z,
    t_max_days=180.0,
)

fig, ax = plt.subplots(figsize=(8.0, 4.6))
ax.set_facecolor("white")
ax.plot(bol.t_days, bol.Lbol, color="#284b83", linewidth=2.2)

ax.set_yscale("log")
ax.set_xlabel("Observer time (days)")
ax.set_ylabel("Bolometric luminosity (erg s$^{-1}$)")
ax.set_title("Forward model: Nickel bolometric light curve")

plt.tight_layout()
plt.show()



In [ ]:
# Multi-band forward example (nickel)
params_nickel_mb = {
    "M_ej": 3.0,
    "v_ej": 1.0,
    "E_Th_in": 1.5,
    "M_Ni": 0.08,
    "R_0": 120.0,
    "x_Ni": 0.2,
    "kappa": 0.12,
    "kappa_gamma": 0.03,
    "T_floor": 3000.0,
}
mb = tf.lightcurve_multiband(
    model="nickel",
    params=params_nickel_mb,
    z=z,
    distance_modulus=distance_modulus,
    filters=filters,
    bands=["B", "V", "R", "I"],
    y_kind="mag",
    mag_system="vega",
    extinction=extinction,
    t_max_days=180.0,
)

palette = {
    "B": "#2563eb",
    "V": "#16a34a",
    "R": "#dc2626",
    "I": "#7c3aed",
}

fig, ax = plt.subplots(figsize=(8.0, 4.6))
ax.set_facecolor("white")
for b in mb.bands:
    ax.plot(mb.t_days, mb.y[b], label=f"{b} band", color=palette.get(b, None), linewidth=2.1)

ax.set_xlim(left=0)
ax.set_ylim(12, 22)
ax.set_xlabel("Observer time (days)")
ax.set_ylabel("Vega magnitude")
ax.invert_yaxis()
ax.set_title("Forward model: Nickel-powered multi-band curves")

leg = ax.legend(loc="upper right", frameon=True)
leg.get_frame().set_alpha(0.92)

plt.tight_layout()
plt.show()



## 4. Prepare Bolometric Data and Fit

`fit_bol` inputs:
- `data`: `tf.BolometricData`
- `model`: model key
- `z`: distance/redshift input
- `priors`: linear and/or log priors
- `fixed`: fixed parameter values
- `sampler`: `emcee`, `zeus`, or `dynesty`
- `sampler_kwargs`: backend settings
- `t_shift`: included in the fit by default; if you want to disable it, set `fixed={"t_shift": 0.0}`

Grid-discretization parameters are handled internally in the default workflow and are not treated as standard user-facing fit parameters.
- `T_floor` is not a bolometric fit parameter; `fit_bol(...)` keeps an internal 1000 K floor only for numerical stability.


In [ ]:
# Load example bolometric data
arr = np.loadtxt(DATA_DIR / "sn1993j_lbol.txt")

# Public APIs expect relative observer-frame days, not raw JD/MJD.
t_days = arr[:, 0] - np.min(arr[:, 0])
Lbol = arr[:, 1]
Lbol_err = arr[:, 2]

data_bol = tf.BolometricData(t_days=t_days, y=Lbol, yerr=Lbol_err)
print("Bolometric data size:", data_bol.t_days.size)


In [ ]:
# Bolometric fit example with emcee
res_bol = tf.fit_bol(
    data=data_bol,
    model="nickel",
    z=z,
    priors={
        "M_ej": (0.7, 5.0),
        "v_ej": (0.2, 3.0),
        "M_Ni": (0.01, 0.2),
        "x_Ni": (0.1, 0.9),
        "t_shift": (-5.0, 5.0),
    },
    fixed={
        "E_Th_in": 0.01,
        "R_0": 10.0,
        "kappa": 0.06,
        "kappa_gamma": 0.03,
    },
    sampler="emcee",
    sampler_kwargs=dict(
        nwalkers=32,
        nsteps=400,
        burnin=100,
        thin=2,
        seed=520,
        progress=False,
        robust_init=False,
    ),
    model_kwargs=dict(
        Nx=24,
        Ny=180,
        t_max_days=140.0,
    ),
)



## 5. Understand `res` (`FitResult`)

Important fields:
- `res.param_names`: sampled/free parameters
- `res.all_param_names`: full parameter order
- `res.fixed`: fixed values used in fit
- `res.samples`: posterior samples `(Ns, ndim)`
- `res.log_prob`: log posterior values
- `res.meta`: metadata (bounds, priors, solver config, etc.)

Recommended accessors:
- `res.best_fit`
- `res.best_params`
- `res.median_params`
- `res.best_log_prob`


In [ ]:
print("model:", res_bol.model)
print("sampler:", res_bol.sampler)
print("param_names:", res_bol.param_names)
print("all_param_names:", res_bol.all_param_names)
print("samples shape:", res_bol.samples.shape)
print("best index:", res_bol.best_index)
print("best log_prob:", res_bol.best_log_prob)




In [ ]:
res_bol.best_fit

## 6. Plot Bolometric Fit and Corner


In [ ]:
fig = tf.plot.fit_bol(
    res_bol,
    data=data_bol,
    show_1sigma=True,
    n_draws=200,
    t_pad=60.0,
)
fig


In [ ]:
# Corner plot for bolometric posterior
try:
    fig_corner_bol = tf.plot.corner(res_bol, max_points=10000)
    fig_corner_bol
except ImportError as e:
    print(e)
    print("Install with: pip install corner")


## 7. Prepare Multi-band Data and Fit

Expected `MultiBandData` format:
- `t_days`: observer-frame days relative to a chosen reference epoch
- `band`: band labels (case-sensitive matching with `filters` keys)
- `y`: magnitude or flux values
- `yerr`: uncertainties in the same observation space as `y`

Fitting rules:
- choose `y_kind="mag"` for magnitudes and `y_kind="flux"` for flux-density data
- most supernova light-curve data are published in magnitudes, so `y_kind="mag"` is the normal choice
- if `y_kind="mag"`, choose `mag_system="ab"` or `mag_system="vega"` explicitly
- `extinction` modifies the model prediction before the comparison step; it does not edit your input data table


In [ ]:
# Load and reshape example multi-band data (wide -> long)
df = pd.read_csv(DATA_DIR / "sn2007gr.csv")

# Convert absolute JD values to relative observer-frame days.
t0 = float(np.nanmin(df["JD"].to_numpy(float)))
df["t_days"] = df["JD"].to_numpy(float) - t0

band_map = [
    ("B", "Bmag", "e_Bmag"),
    ("V", "Vmag", "e_Vmag"),
    ("R", "Rmag", "e_Rmag"),
    ("I", "Imag", "e_Imag"),
]

rows = []
for band, mag_col, err_col in band_map:
    if mag_col not in df.columns or err_col not in df.columns:
        continue

    y = pd.to_numeric(df[mag_col], errors="coerce").to_numpy(float)
    yerr = pd.to_numeric(df[err_col], errors="coerce").to_numpy(float)
    t = df["t_days"].to_numpy(float)

    mask = np.isfinite(t) & np.isfinite(y) & np.isfinite(yerr) & (yerr > 0)
    if not np.any(mask):
        continue

    rows.append(
        pd.DataFrame(
            {
                "t_days": t[mask],
                "band": np.full(np.sum(mask), band, dtype=object),
                "y": y[mask],
                "yerr": yerr[mask],
            }
        )
    )

lc = pd.concat(rows, ignore_index=True).sort_values("t_days").reset_index(drop=True)
# Optional cut for tutorial speed
lc = lc[lc["t_days"] < 110].reset_index(drop=True)

data_mb = tf.MultiBandData(
    t_days=lc["t_days"].to_numpy(float),
    band=lc["band"].to_numpy(dtype=object),
    y=lc["y"].to_numpy(float),
    yerr=lc["yerr"].to_numpy(float),
)

print("Multiband data size:", data_mb.t_days.size)
lc.head()


In [ ]:
# Multi-band fit example with emcee
res_mb = tf.fit_multiband(
    data=data_mb,
    model="nickel",
    z=z,
    distance_modulus=distance_modulus,
    filters=filters,
    y_kind="mag",
    mag_system="vega",
    extinction=extinction,
    priors={
        "M_ej": (1.0, 5.0),
        "v_ej": (0.3, 3.0),
        "M_Ni": (0.01, 0.5),
        "t_shift": (-2.0, 2.0),
    },
    fixed={
        "E_Th_in": 0.01,
        "R_0": 10.0,
        "x_Ni": 0.5,
        "kappa": 0.06,
        "kappa_gamma": 0.03,
        "T_floor": 4500.0,
    },
    sampler="emcee",
    sampler_kwargs=dict(
        nwalkers=32,
        nsteps=400,
        burnin=100,
        thin=2,
        seed=123,
        progress=False,
        robust_init=False,
    ),
    model_kwargs=dict(
        Nx=20,
        Ny=160,
        t_max_days=180.0,
    ),
)



In [ ]:
print("model:", res_mb.model)
print("sampler:", res_mb.sampler)
print("param_names:", res_mb.param_names)
print("samples shape:", res_mb.samples.shape)

res_mb.best_fit


## 8. Plot Multi-band Fit and Corner


In [ ]:
fig_mb = tf.plot.fit_multiband(
    res_mb,
    data=data_mb,
    show_1sigma=True,
    n_draws=200,
    t_pad=40.0,
)
fig_mb


In [ ]:
# Corner plot for multi-band posterior
try:
    fig_corner_mb = tf.plot.corner(res_mb, max_points=10000)
    fig_corner_mb
except ImportError as e:
    print(e)
    print("Install with: pip install corner")



## 9. Custom Plotting (Minimal)

If you do not want to use `tf.plot.*`, the minimal workflow is:
1. Read best-fit parameters directly: `params_best = res.best_params`.
2. Prepare the forward inputs you need directly: `z` for bolometric, and `z` plus `filters` for multi-band.
3. When needed, pass an explicit distance (`distance_modulus`) and either direct `A_band` values or structured dust-law `extinction`.
4. Call `tf.predict_bol(...)` or `tf.predict_multiband(...)` to compute theoretical curves.
5. Plot model + observed data with your own matplotlib style.

`tf.predict_multiband(...)` still returns the observation space you request, but internally the model is evaluated in `f_nu` before extinction and magnitude conversion are applied.


In [ ]:
# Step 1: read best-fit model input directly
params_best = res_bol.best_params
t_shift_best = res_bol.best_t_shift

# Step 2: compute theoretical curve on a custom time grid
t_line = np.linspace(-2.0, 120.0, 400)
L_line = tf.predict_bol(
    model=res_bol.model,
    params=params_best,
    z=z,
    t_days=t_line,
)

# Step 3: plot data + model and tune style
fig, ax = plt.subplots(figsize=(8, 6))
ax.errorbar(
    data_bol.t_days, data_bol.y, yerr=data_bol.yerr,
    fmt="o", ms=6, mfc="#E6E6E6", mec="#2F2F2F", ecolor="#2F2F2F",
    capsize=2, label="data"
)
ax.plot(t_line - t_shift_best, L_line, color="#4D4D4D", lw=2.2, label="best-fit model")
ax.set_yscale("log")
ax.set_xlabel("Time (days)")
ax.set_ylabel("Bolometric Luminosity (erg s$^{-1}$)")
ax.legend(frameon=False)
ax.minorticks_on()
fig.tight_layout()


### Multi-band Minimal Example

Use the same 3-step flow for multi-band data: read best-fit parameters, compute model curves, then style the figure.


In [ ]:
# Step 1: read best-fit model input directly
params_mb = res_mb.best_params
t_shift_mb = res_mb.best_t_shift

# Step 2: compute model curves for each band
bands = list(dict.fromkeys(data_mb.band.tolist()))
t_line = np.linspace(0, 150, 300)
colors = dict(zip(bands, plt.cm.tab10(np.linspace(0, 1, len(bands)))))

fig, ax = plt.subplots(figsize=(8, 6))
for b in bands:
    m = (data_mb.band == b)
    y_line = tf.predict_multiband(
        model=res_mb.model,
        params=params_mb,
        z=z,
        distance_modulus=distance_modulus,
        filters=filters,
        t_days=t_line,
        band=np.array([b] * len(t_line), dtype=object),
        y_kind="mag",
        mag_system="vega",
        extinction=extinction,
    )
    ax.errorbar(
        data_mb.t_days[m], data_mb.y[m], yerr=data_mb.yerr[m],
        fmt="o", ms=4, mfc="white", mec=colors[b], ecolor=colors[b],
        capsize=2, alpha=0.9, label=f"{b} data"
    )
    ax.plot(t_line - t_shift_mb, y_line, color=colors[b], lw=2.0, label=f"{b} model")

# Step 3: optimize figure style
ax.set_xlabel("Time (days)")
ax.set_ylabel("Vega mag")
ax.set_ylim(18, 12)
ax.legend(ncol=2, frameon=False, fontsize=9)
ax.minorticks_on()
fig.tight_layout()


## 10. Save and Load


In [ ]:
out_path = tf.save(res_mb, path=Path("mcmc_out") / "fit_nickel_multiband_demo.npz")
loaded = tf.load(out_path)

print("saved to:", out_path)
print("loaded model:", loaded["model"])
print("loaded sampler:", loaded["sampler"])
print("loaded samples shape:", loaded["samples"].shape)
